In [75]:
train_df = pd.read_csv('/content/sample_data/train.csv')
test_df = pd.read_csv('/content/sample_data/test.csv')

In [24]:
! pip install transformers[torch]
! pip install evaluate
! pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [76]:
train_df['label'] = train_df['label'].astype("category")
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284 entries, 0 to 283
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   text         284 non-null    object  
 1   label        284 non-null    category
 2   text_tokens  284 non-null    object  
 3   text_wosw    284 non-null    object  
 4   text_lemmas  284 non-null    object  
dtypes: category(1), object(4)
memory usage: 9.9+ KB


In [77]:
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
import evaluate
import numpy as np

In [78]:
# Chargement des données
dtypes = {
    'label': 'category',
    'text_tokens': 'object'
}



#  Découpage train / validation
from sklearn.model_selection import train_test_split
train_df, dev_df = train_test_split(train_df, test_size=0.2, random_state=42, shuffle=True)
train_df.reset_index(drop=True, inplace=True)
dev_df.reset_index(drop=True, inplace=True)

# Création des identifiants de classes
class_names = sorted(train_df.label.unique().categories.to_list())
label2id = {class_names[i]: i for i in range(len(class_names))}
id2label = {i: class_names[i] for i in range(len(class_names))}

train_df['label'] = train_df.label.map(label2id)
dev_df['label'] = dev_df.label.map(label2id)

# Transformation en Dataset HuggingFace
target_columns = ["text_tokens", "label"]
train_ds = Dataset.from_pandas(train_df[target_columns], split="train")
dev_ds = Dataset.from_pandas(dev_df[target_columns], split="validation")
data = DatasetDict({"train": train_ds, "validation": dev_ds})

In [50]:
data['train'][0]

{'text_tokens': 'Le Corbusier demeure une figure incontournable de l architecture moderne . Ses théories sur la maison comme machine à habiter ont profondément transformé notre manière de concevoir les espaces de vie . Entre béton brut et lumière savamment dosée , son oeuvre continue d inspirer des générations d architectes à travers le monde entier .',
 'label': 0}

In [79]:
data['train'].features

{'text_tokens': Value('string'), 'label': Value('int64')}

In [80]:

# Tokenisation
model_ckpt = "almanach/camembert-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess(dataset):
    return tokenizer(dataset['text_tokens'], padding=True, truncation=True)

tokenized_data = data.map(preprocess, batched=True)

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Map:   0%|          | 0/57 [00:00<?, ? examples/s]

In [81]:
tokenizer.vocab_size

32005

In [61]:
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoModel
from datasets import Features, Value, ClassLabel, Dataset, DatasetDict

In [55]:
tokenizer.model_max_length

512

In [82]:


#  Evaluation
accuracy = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1-macro": f1["f1"]}




In [83]:
batch_size = 32
training_args = TrainingArguments(
    # dossier pour le sauvegarde du modèle. On peut récupérer le modèle affiné
    output_dir=f"{model_ckpt}-finetuned-med",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    # nb total d'époques
    num_train_epochs=10,
    weight_decay=0.01,
    # évaluer à chaque époque
    eval_strategy="epoch",
    # sauvegarder le modèle, à la fin de chaque époque.
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
)

In [84]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [85]:
def init_trainer():
  model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt, num_labels=len(class_names), id2label=id2label, label2id=label2id
    ).to(device)
  return Trainer(
      model=model,
      args=training_args,
      train_dataset=tokenized_data["train"],
      eval_dataset=tokenized_data["validation"],
      #tokenizer=tokenizer,
      data_collator=data_collator,
      compute_metrics=compute_metrics,
  ), model

In [86]:
trainer, model = init_trainer()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: almanach/camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1-macro
1,No log,2.607419,0.192982,0.051101
2,No log,2.579194,0.245614,0.134797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [39]:
trainer, model = init_trainer()

# 🔹 Lancement de l'affinage
trainer.train()


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForSequenceClassification LOAD REPORT from: almanach/camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' a

Epoch,Training Loss,Validation Loss,Accuracy,F1-macro
1,No log,2.625483,0.166667,0.106061
2,No log,2.620329,0.083333,0.019231
3,No log,2.616214,0.083333,0.019231
4,No log,2.611157,0.083333,0.019231
5,No log,2.606939,0.083333,0.019231
6,No log,2.603586,0.083333,0.019231
7,No log,2.601125,0.083333,0.019231
8,No log,2.598837,0.083333,0.019231
9,No log,2.597480,0.083333,0.019231
10,No log,2.596850,0.083333,0.019231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=20, training_loss=2.5977266311645506, metrics={'train_runtime': 940.8591, 'train_samples_per_second': 0.478, 'train_steps_per_second': 0.021, 'total_flos': 32378481324000.0, 'train_loss': 2.5977266311645506, 'epoch': 10.0})

In [46]:
from sklearn.model_selection import train_test_split

In [44]:
import sklearn


train_df

In [48]:
train_df

,text,label,text_tokens,text_wosw,text_lemmas
0,Redécouvrir Marguerite Yourcenar en 2024 nest ...,6,Redécouvrir Marguerite Yourcenar en 2024 nest ...,Redécouvrir Marguerite Yourcenar 2024 nest exe...,redécouvrir Marguerite Yourcenar en 2024 nest ...
1,Le festival d'Avignon reste chaque été le rend...,13,Le festival d' Avignon reste chaque été le ren...,festival Avignon été rendez-vous incontournabl...,le festival de avignon reste chaque être le re...
2,La Biennale de Design de Saint-Étienne a une f...,5,La Biennale de Design de Saint-Étienne a une f...,Biennale Design Saint-Étienne fois prouvé capa...,le biennale de Design de Saint-Étienne avoir u...
3,Le Corbusier demeure une figure incontournable...,0,Le Corbusier demeure une figure incontournable...,Corbusier demeure figure incontournable l arch...,le corbusier demeure un figure incontournable ...
4,Le musée Rodin de Paris accueille chaque année...,12,Le musée Rodin de Paris accueille chaque année...,musée Rodin Paris accueille année milliers vis...,le musée Rodin de Paris accueille chaque année...
5,Le ballet classique connaît une renaissance in...,4,Le ballet classique connaît une renaissance in...,ballet classique connaît renaissance inattendu...,le ballet classique connaître un renaissance i...
6,La sculpture occupe une place singulière dans ...,12,La sculpture occupe une place singulière dans ...,sculpture occupe place singulière lhistoire la...,le sculpture occuper un place singulier dans l...
7,Les prix littéraires de la rentrée ont cette a...,6,Les prix littéraires de la rentrée ont cette a...,prix littéraires rentrée année mis lumière aut...,le prix littéraire de le rentrée avoir ce anné...
8,Tokyo présente un visage architectural unique ...,0,Tokyo présente un visage architectural unique ...,Tokyo présente visage architectural unique mon...,tokyo présente un visage architectural unique ...
9,Le mobilier scandinave reste une référence inc...,5,Le mobilier scandinave reste une référence inc...,mobilier scandinave référence incontournable m...,le mobilier scandinave reste un référence inco...


In [47]:
train_df, dev_df = sklearn.model_selection.train_test_split(train_df, test_size=0.20, train_size=0.80, random_state=42, shuffle=True, stratify=df.labels)

AttributeError: 'DataFrame' object has no attribute 'labels'